# 3. pathway-supervised contrastive fine-tuning

GPU runtime

1. Learning-rate selection on the validation set
2. Two more seeds with the chosen learning rate
3. Evaluation of the three seeds together with the baselines
4. Mean-centered evaluation
5. Ablation of explicit negatives

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os

REPO = 'https://github.com/GITHUB_ID/pathway-supervised-embedder.git'
BASE = '/content/drive/MyDrive/kegg-bge'
DATA = '/content/data/splits'

if os.path.isdir(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
S = f'{CODE}/scripts'
!mkdir -p /content/data {BASE}
!cp -r {BASE}/data/splits /content/data/
def latest(d): return sorted(glob.glob(f'{d}/run-*'))[-1]

In [ ]:
OUTB = f'{BASE}/result_bienc'
EMB  = f'{BASE}/embeddings'
EVAL = f'{BASE}/eval'
RUNS_FILE = f'{BASE}/bienc_runs.json'
runs = json.load(open(RUNS_FILE)) if os.path.exists(RUNS_FILE) else {}
def save_runs():
    json.dump(runs, open(RUNS_FILE, 'w'), indent=2)
COMMON = '--bf16 --eval-every 50 --epochs 1'

## 1. Learning rate


In [ ]:
sweep = {}
for lr in ['5e-6', '1e-5', '2e-5']:
    !python {S}/train_biencoder.py --data-dir {DATA} --out-dir {OUTB}/sweep {COMMON} --lr {lr} --seed 42
    run = latest(f'{OUTB}/sweep')
    info = json.load(open(f'{run}/best_info.json'))
    sweep[lr] = (run, info['val_pathway_map'], info['step'])
    print(f'lr {lr}: best val pathway mAP {info["val_pathway_map"]:.4f} at step {info["step"]}')
BEST_LR = max(sweep, key=lambda k: sweep[k][1])
runs.update({'lr': BEST_LR, 'kegg_bge_s42': sweep[BEST_LR][0], 'sweep': {k: v[0] for k, v in sweep.items()}})
save_runs(); print('chosen lr:', BEST_LR)

## 2. Two more seeds

In [ ]:
for seed in [1, 2]:
    !python {S}/train_biencoder.py --data-dir {DATA} --out-dir {OUTB}/seeds {COMMON} --lr {runs['lr']} --seed {seed}
    runs[f'kegg_bge_s{seed}'] = latest(f'{OUTB}/seeds')
save_runs(); print(json.dumps(runs, indent=1))

## 3. Evaluation with the baselines


In [ ]:
custom = ' '.join(f'{k}={runs[k]}/best' for k in ['kegg_bge_s42', 'kegg_bge_s1', 'kegg_bge_s2'])
!python {S}/embed.py --data-dir {DATA} --out-dir {EMB} --bf16 --custom {custom}
CROSS = sorted(glob.glob(f'{BASE}/result_crossencoder/run-*'))
pair = f'--pair-preds cross_encoder={CROSS[-1]}/test_predictions.jsonl' if CROSS else ''
!pip -q install tabulate
!python {S}/evaluate.py --data-dir {DATA} --emb-dir {EMB} --out-dir {EVAL} --reference kegg_bge_s42 {pair}

import pandas as pd
t = pd.read_csv(f'{EVAL}/all_variants_test.csv', index_col=0)
choice = json.load(open(f'{EVAL}/pooling_choice.json'))
cols = ['category_map', 'subcategory_map', 'pathway_map', 'pathway_ndcg10', 'pathway_silhouette', 'pair_auroc', 'pair_auroc_pos_vs_hard']
k = t.loc[[choice[f'kegg_bge_s{s}'] for s in (42, 1, 2)], cols]
display(t.loc[[choice['bge']], cols].round(4))
print('KEGG-BGE mean ± SD over seeds:')
print((k.mean().round(3).astype(str) + ' ± ' + k.std().round(3).astype(str)).to_string())
from IPython.display import Markdown
Markdown(open(f'{EVAL}/report_test.md').read())

## 4. Mean-centered embeddings

In [ ]:
!python {S}/evaluate.py --data-dir {DATA} --emb-dir {EMB} --out-dir {BASE}/eval_centered --reference kegg_bge_s42 --center
Markdown(open(f'{BASE}/eval_centered/report_test.md').read())

## 5. Ablation of explicit negatives


In [ ]:
for neg in ['none', 'hard', 'soft']:
    !python {S}/train_biencoder.py --data-dir {DATA} --out-dir {OUTB}/ablation_{neg} {COMMON} --lr {runs['lr']} --seed 42 --neg-types {neg}
    runs[f'kegg_bge_{neg}'] = latest(f'{OUTB}/ablation_{neg}')
save_runs()
custom = ' '.join(f'kegg_bge_{n}={runs[f"kegg_bge_{n}"]}/best' for n in ['none', 'hard', 'soft'])
!python {S}/embed.py --data-dir {DATA} --out-dir {EMB} --bf16 --splits test --custom {custom}
!python {S}/evaluate.py --data-dir {DATA} --emb-dir {EMB} --out-dir {BASE}/eval_ablation --reference kegg_bge_s42 --pooling-from {EVAL}/pooling_choice.json --select-split test
t = pd.read_csv(f'{BASE}/eval_ablation/all_variants_test.csv', index_col=0)
rows = [v for v in t.index if v.split('__')[0] in ('kegg_bge_none', 'kegg_bge_hard', 'kegg_bge_soft', 'kegg_bge_s42') and v.endswith('__cls')]
display(t.loc[rows, ['category_map', 'subcategory_map', 'pathway_map', 'pathway_silhouette', 'pair_auroc_pos_vs_hard']].round(3))